In [1]:
import sys
from pathlib import Path
import geopandas as gpd
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import shutil

# Find the repository root.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, photo_paths
from src.prep import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    ground_to_mercator_extent,
    update_tracker,
)

cfg = load_config()

In [4]:
INDEX_PATH = cfg["paths"]["index"]
print("Source index:", INDEX_PATH)
print("County:", cfg["county"])

Source index: C:\Users\eu01251115\LocalProjects\batch_georef\data\main_index_blank.gpkg
County: Carver


In [5]:
# Load the main index.
LAYER = "photos"
index_gdf = gpd.read_file(INDEX_PATH, layer=LAYER)

### Examine and choose available years and projects
The first cell shows all available years for the chosen county. 
second cell below displays unique combos between flight codes and 

In [8]:
# examine all years in the county

county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

print(cfg["county"], "county flight years:")

for year in sorted(county_gdf["year"].dropna().unique()):
    print(year)

Carver county flight years:
1937
1938
1940
1947
1951
1953
1955
1956
1957
1960
1962
1963
1964
1967
1969
1971


In [7]:
# Examine unique year + flight code combinations for the county
county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

flights = (
    county_gdf[["year", "flight_code", "county"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["year", "flight_code", "county"])
)

display(flights)

,year,flight_code,county
13511,1937,Carver,Carver
13788,1937,Hennepin,Carver
13773,1937,Scott,Carver
13804,1938,bjm,Carver
13809,1940,Carver,Carver
13886,1940,McLeod,Carver
13894,1940,Scott,Carver
13897,1947,dv,Carver
14059,1951,Carver,Carver
14173,1951,Scott,Carver


### Manually determining ground scale:

- Visit the [MHAPO Air Photo Holdings page](https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery). 
- Select your chosen county to open a PDF with metadata. 
- Compare this to the list above to determine the ground scale for your target year or years. Currently, this tool supports processing multiple years at the same time, but only if they share the same ground scale. 
- **Choose a year or multiple years with the same scale**
- Specify the year(s) and scale for your current batch in config.yml.
- The batch will be named for the county, years, and scale. The photos and the folders they are in will be named for the county and year. 

#### Once you have configured your year(s) and scale in config.yml, restart the kernel and start again from the beginning.


In [9]:
# Select records for the configured county and years.
mask = (
    index_gdf["county"].str.casefold().eq(
        cfg["county"].casefold()
    )
    & index_gdf["year"].isin(cfg["years"])
)

In [10]:
batch_gdf = index_gdf.loc[mask].copy()

In [11]:
TRACKER_PATH = cfg["paths"]["tracker"]
LAYER = "photos"
print("Path for tracker:", TRACKER_PATH)
print("County:", cfg["county"])
print("Years:", cfg["years"])
print("Scale:", cfg["scale"])

Path for tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1963_20000_tracker.gpkg
County: Carver
Years: [1963]
Scale: 20000


## Create a tracker table

In [12]:
if batch_gdf.empty:
    raise ValueError("No photographs match this batch configuration.")

if (
    batch_gdf["cty_year_id"].isna().any()
    or not batch_gdf["cty_year_id"].is_unique
):
    raise ValueError("Batch contains missing or duplicate photo IDs.")

In [13]:
# check length and confirm CRS
print(f"Selected {len(batch_gdf)} photographs")
print("CRS:", batch_gdf.crs)

Selected 100 photographs
CRS: EPSG:3857


In [14]:
# Assign the configured scale to all selected photographs.
batch_gdf["scale"] = cfg["scale"]

In [15]:
# preview it
display(
    batch_gdf[
        ["cty_year_id", "county", "year", "scale", "url"]
    ].head()
)

,cty_year_id,county,year,scale,url
14371,carver_1963_wm-1dd-3,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-3.jpg
14372,carver_1963_wm-1dd-7,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-7.jpg
14373,carver_1963_wm-1dd-9,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-9.jpg
14374,carver_1963_wm-1dd-11,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-11...
14375,carver_1963_wm-1dd-13,Carver,1963,20000,https://geo.lib.umn.edu/carver/y1963/wm-1dd-13...


In [16]:
batch_gdf.columns

Index(['cty_year_id', 'status', 'county', 'year', 'photo_id', 'ground_width',
       'ground_height', 'wm_width', 'wm_height', 'scale', 'dpi_x', 'dpi_y',
       'width_px', 'height_px', 'worldfile_json', 'url', 'flight_code',
       'source_id', 'date_prepared', 'date_georef', 'gcp_count', 'confidence',
       'error', 'latitude', 'geometry'],
      dtype='str')

In [17]:
if TRACKER_PATH.exists():
    raise FileExistsError(
        f"Batch tracker already exists: {TRACKER_PATH}"
    )

TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)

batch_gdf.to_file(
    TRACKER_PATH,
    layer=LAYER,
    driver="GPKG",
)

print(f"Created batch tracker: {TRACKER_PATH}")

FileExistsError: Batch tracker already exists: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1963_20000_tracker.gpkg

## Download JPEGS and calculate extents

In [15]:
# Load the batch tracker for preparation
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print(f"Batch: {cfg['batch_name']}")
print(f"Photographs: {len(tracker_gdf)}")

display(tracker_gdf.head())

Batch: carver_1951_20000
Photographs: 273


,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1951_wm_01h_014,None,Carver,1951,wm_01h_014,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14243,None,None,None,None,None,44.935001,POINT (-10464154.43 5611294.584)
1,carver_1951_wm_01h_018,None,Carver,1951,wm_01h_018,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14245,None,None,None,None,None,44.963402,POINT (-10464945.132 5615761.688)
2,carver_1951_wm_01h_019,None,Carver,1951,wm_01h_019,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14246,None,None,None,None,None,44.963299,POINT (-10462940.78 5615745.483)
3,carver_1951_wm_01h_034,None,Carver,1951,wm_01h_034,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14247,None,None,None,None,None,44.878799,POINT (-10465056.385 5602461.071)
4,carver_1951_wm_01h_035,None,Carver,1951,wm_01h_035,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14248,None,None,None,None,None,44.878601,POINT (-10462841.417 5602429.918)


In [16]:
# Download original JPEGs.
download_results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Downloading photographs",
):
    photo_id = str(photo["cty_year_id"])
    jpeg_path = photo_paths(photo, cfg)["jpeg"]

    try:
        download_photo(photo["url"], jpeg_path)

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": None,
        })

    except Exception as exc:
        print(f"FAILED: {photo_id}: {exc}")

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": str(exc),
        })

download_df = pd.DataFrame(download_results)

print(f"Successful downloads: {download_df['error'].isna().sum()}")
print(f"Failed downloads: {download_df['error'].notna().sum()}")

display(download_df.head())

Successful downloads: 273
Failed downloads: 0


,cty_year_id,jpeg_path,error
0,carver_1951_wm_01h_014,C:\Users\eu01251115\LocalProjects\batch_georef...,None
1,carver_1951_wm_01h_018,C:\Users\eu01251115\LocalProjects\batch_georef...,None
2,carver_1951_wm_01h_019,C:\Users\eu01251115\LocalProjects\batch_georef...,None
3,carver_1951_wm_01h_034,C:\Users\eu01251115\LocalProjects\batch_georef...,None
4,carver_1951_wm_01h_035,C:\Users\eu01251115\LocalProjects\batch_georef...,None


In [17]:
# Collect preparation results for each photo.
results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Calculating extents",
):
    photo_id = photo["cty_year_id"]
    image_path = photo_paths(photo, cfg)["jpeg"]

    result = {
        "cty_year_id": photo_id,
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # Read metadata from the existing JPEG.
        metadata = get_image_metadata(image_path)
        result.update(metadata)

        # Calculate ground dimensions in meters.
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

        # Convert dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

        # Mark preparation complete only after successful calculations.
        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

    except Exception as e:
        result["error"] = str(e)
        print(f"FAILED: {photo_id}: {e}")

    results.append(result)

# Convert results to a DataFrame for inspection.
results_df = pd.DataFrame(results)

print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

display(results_df.head())

Calculating extents: 100%|██████████| 273/273 [00:00<00:00, 1720.24it/s]

FAILED: carver_1951_wm_01h_014: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_018: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_019: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_034: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_035: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_02h_061: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_02h_062: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_003: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_004: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_005: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_006: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_007: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_008: Width, DPI, and scale must be numeric.
FAILED: carver_1951_wm_01h_009: Width, DPI, and scale must be numeric.
FAILED

,cty_year_id,width_px,height_px,dpi_x,dpi_y,ground_width,ground_height,wm_width,wm_height,date_prepared,error
0,carver_1951_wm_01h_014,5418,5335,NaN,NaN,NaN,NaN,NaN,NaN,None,"Width, DPI, and scale must be numeric."
1,carver_1951_wm_01h_018,5418,5335,NaN,NaN,NaN,NaN,NaN,NaN,None,"Width, DPI, and scale must be numeric."
2,carver_1951_wm_01h_019,5418,5335,NaN,NaN,NaN,NaN,NaN,NaN,None,"Width, DPI, and scale must be numeric."
3,carver_1951_wm_01h_034,5451,5352,NaN,NaN,NaN,NaN,NaN,NaN,None,"Width, DPI, and scale must be numeric."
4,carver_1951_wm_01h_035,5451,5369,NaN,NaN,NaN,NaN,NaN,NaN,None,"Width, DPI, and scale must be numeric."


In [18]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

Updated 273 records.


### Final load & preview

In [19]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf

,cty_year_id,status,county,year,photo_id,ground_width,ground_height,wm_width,wm_height,scale,...,url,flight_code,source_id,date_prepared,date_georef,gcp_count,confidence,error,latitude,geometry
0,carver_1951_wm_01h_014,failed_prep,Carver,1951,wm_01h_014,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14243,NaN,None,None,None,"Width, DPI, and scale must be numeric.",44.935001,POINT (-10464154.43 5611294.584)
1,carver_1951_wm_01h_018,failed_prep,Carver,1951,wm_01h_018,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14245,NaN,None,None,None,"Width, DPI, and scale must be numeric.",44.963402,POINT (-10464945.132 5615761.688)
2,carver_1951_wm_01h_019,failed_prep,Carver,1951,wm_01h_019,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14246,NaN,None,None,None,"Width, DPI, and scale must be numeric.",44.963299,POINT (-10462940.78 5615745.483)
3,carver_1951_wm_01h_034,failed_prep,Carver,1951,wm_01h_034,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14247,NaN,None,None,None,"Width, DPI, and scale must be numeric.",44.878799,POINT (-10465056.385 5602461.071)
4,carver_1951_wm_01h_035,failed_prep,Carver,1951,wm_01h_035,NaN,NaN,NaN,NaN,20000,...,https://geo.lib.umn.edu/dnr_historical_aerial_...,wm,14248,NaN,None,None,None,"Width, DPI, and scale must be numeric.",44.878601,POINT (-10462841.417 5602429.918)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,carver_1951_wm-4h-54,prepared,Carver,1951,wm-4h-54,4687.146667,4649.046667,6591.258193,6537.680408,20000,...,https://geo.lib.umn.edu/carver/y1951/wm-4h-54.jpg,Carver,92234,2026-11-18,None,None,None,NaN,44.674242,POINT (-10436003.668 5570382.52)
269,carver_1951_wm-4h-56,prepared,Carver,1951,wm-4h-56,4680.373333,4662.593333,6581.708268,6556.705396,20000,...,https://geo.lib.umn.edu/carver/y1951/wm-4h-56.jpg,Carver,92235,2026-11-18,None,None,None,NaN,44.674022,POINT (-10440535.707 5570348.081)
270,carver_1951_wm-4h-58,prepared,Carver,1951,wm-4h-58,4693.920000,4649.046667,6600.731308,6537.629078,20000,...,https://geo.lib.umn.edu/carver/y1951/wm-4h-58.jpg,Carver,92236,2026-11-18,None,None,None,NaN,44.673787,POINT (-10445067.635 5570311.294)
271,carver_1951_wm-4h-60,prepared,Carver,1951,wm-4h-60,4666.826667,4649.046667,6562.603538,6537.600876,20000,...,https://geo.lib.umn.edu/carver/y1951/wm-4h-60.jpg,Carver,92237,2026-11-18,None,None,None,NaN,44.673537,POINT (-10449599.563 5570272.159)
